# CF0360 all-species DA, May 2019: where are the analysis increments suspiciously large?

This notebook flags every land tile and 3-hourly assimilation cycle of the all-species DA run
(`M21C_test_CF0360_allsp_superob025ml_OL6sc_xc03125`, 1–31 May 2019, 247 cycles) whose **analysis increment**
(analysis − forecast) looks excessive, and maps where that happens.

**Notation used in all figure titles:**

- **Δx** = `x_ANA − x_FCST`, the analysis increment of state x at one assimilation cycle (every 3 h), ensemble mean.
- **σf(x)** = `x_FCST_ENSSTD`, the forecast ensemble standard deviation of x at that cycle, i.e. the spread the EnKF sees
  when it makes the update.
- **z = |Δx| / σf(x)**: the increment in units of forecast spread.
- **SFMC** / **RZMC**: surface (0–5 cm) / root-zone (0–100 cm) soil moisture [m³/m³]. **TSURF**: surface temperature [K].
- **dW** = `SRFEXC_INCR + RZEXC_INCR − CATDEF_INCR`: profile water increment of the Catchment model [mm].
- **snow update** = the tile had a non-zero SWE increment (`WESNN1+2+3_INCR ≠ 0`) in that cycle.
- **ΔTC** = the largest |increment| of the three Catchment surface temperatures (`TCFSAT`, `TCFTRN`, `TCFWLT`) [K].

**The criteria** (agreed thresholds). Each is evaluated per tile and per cycle:

| Group | Label | Flag when | Why it is suspicious |
|---|---|---|---|
| A: large vs spread | `A_RZ` | σf > 0 **and** z(RZMC) > 3 **and** \|ΔRZMC\| > 0.02 | an EnKF update should rarely move the state by more than ~2–3 forecast std (the gain is ≤ 1) |
| | `A_SF` | σf > 0 **and** z(SFMC) > 3 **and** \|ΔSFMC\| > 0.05 | same, surface layer |
| | `A_TS` | σf > 0 **and** z(TSURF) > 3 **and** \|ΔTSURF\| > 2 K **and no snow update** | same, surface temperature |
| | `A_TS_snow` | as `A_TS`, but **with** a snow update in the same cycle | the TSURF change comes with the MODIS SCF snow update, which I believe is rule-based (not EnKF), so the spread test does not really apply; kept separate so it does not hide the Tb-driven cases |
| B: physically large | `B_dW` | \|dW\| > 100 mm | a large amount of water added/removed in one 3-h update |
| | `B_RZ` | \|ΔRZMC\| > 0.05 | |
| | `B_SF` | \|ΔSFMC\| > 0.1 | |
| | `B_TC` | ΔTC > 5 K | |
| C: pinned at saturation | `C_pin` | RZMC_ANA within 0.02 of porosity **and** σf(RZMC) < 0.005 for ≥ 5 days in a row (40 cycles) | state stuck at saturation with no spread left to be corrected |
| | `C_pin_DA` | `C_pin` **and** the tile had at least one wetting `A_RZ`, `B_dW` or `B_RZ` flag during the month | pinned tiles that DA pushed there (naturally saturated wetlands are otherwise also flagged) |

When writing this notebook, the top TSURF events were all snow tiles with an almost-zero TSURF spread and a MODIS SCF
obs of snow where the model had none (O−F = +1). That is why snow-update cycles are split off into `A_TS_snow`. The minimum absolute sizes in group A stop tiles with an almost-zero spread from being flagged on tiny increments.
The absolute SFMC minimum (0.05) and the TSURF thresholds (2 K for A, 5 K for B) were set here. They are easy to change in the configuration cell.

**Obs-space companion (D).** For every assimilated obs, the normalized innovation `(O − F) / sqrt(obsvar + fcstvar)`.
We count obs with |value| > 5 per tile (the tile the obs was assigned to). Large increments near large normalized
innovations point at the obs or their QC. Large increments without them point at the filter (covariances, spread).

**How values are mapped.** For each land tile we count its flagged cycles (0–247). Each **cube cell** then shows the
**largest count among its land tiles** (the worst tile, not an average, so a single bad tile is not diluted).
Cells are drawn on their true shapes from the BCS cell raster (see `cf0360_grid_and_tiles.ipynb`). **Grey** = land with no
flagged cycle, white = ocean.

Run on Discover, GEOSpyD Python 3.13. It reads ~740 files (a few minutes), and the per-tile counts are cached.

## 1. Imports and configuration

In [ ]:
import glob
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature

%matplotlib inline


def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())

ROOT = Path('/gpfsm/dnb06/projects/p284/M21C_testing')
EXP = 'M21C_test_CF0360_allsp_superob025ml_OL6sc_xc03125'
OUT = ROOT / EXP / 'output' / 'CF0360x6C_GLOBAL'
START, END = '20190501', '20190531'           # cycles whose date is within [START, END]

BCS = Path('/discover/nobackup/projects/gmao/bcs_shared/fvInput/ExtData/esm/tiles/v14')
BCS_GEOM = BCS / 'geometry' / 'CF0360x6C_CF0360x6C'
CATCH_PARAMS = BCS / 'land' / 'CF0360x6C' / 'clsm' / 'catch_params.nc4'
N_FACE = 360

CACHE_DIR = ROOT / 'analysis_M21C_land' / 'allsp_suspicious_increments_may2019'
FIG_DIR = REPO_ROOT / 'projects' / 'M21C_land' / 'figures' / 'allsp_suspicious_increments_may2019'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RECOMPUTE = False

# --- thresholds (see the table above) ---------------------------------------------------
Z_MAX = 3.0                  # group A: |increment| / forecast ensemble std
MIN_DRZ_A = 0.02             # group A: minimum |ΔRZMC| [m3/m3]
MIN_DSF_A = 0.05             # group A: minimum |ΔSFMC| [m3/m3]
MIN_DTS_A = 2.0              # group A: minimum |ΔTSURF| [K]
MAX_DW = 100.0               # group B: |dW| [mm]
MAX_DRZ = 0.05               # group B: |ΔRZMC| [m3/m3]
MAX_DSF = 0.10               # group B: |ΔSFMC| [m3/m3]
MAX_DTC = 5.0                # group B: max |ΔTC| [K]
PIN_TOL = 0.02               # group C: porosity - RZMC_ANA below this [m3/m3]
PIN_MAX_SPREAD = 0.005       # group C: σf(RZMC) below this [m3/m3]
PIN_CYCLES = 40              # group C: consecutive cycles (5 days x 8)
MAX_NORM_INNOV = 5.0         # group D: |(O-F)/sqrt(obsvar+fcstvar)|

N_TOP = 20                   # events per criterion in the top-events table (section 6)

## 2. Geometry and porosity

- **Tile table** (BCS): land tiles come first, and land tile *k* is row *k*. That is the tile order of the output
  files and of ObsFcstAna `tilenum` (1-based). For each tile we need its cube cell (`i_indg`, `j_indg`) and its
  centre (for the event table).
- **Porosity** (`POROS`, BCS `catch_params.nc4`): the volumetric soil moisture at saturation, per tile. It ranges
  from 0.37 (mineral soils) to 0.93 (peat).
- **Cube-cell raster**: every 20th pixel of the 30″ cell raster (~0.17°), for drawing cells on their true shapes.

In [ ]:
with nc.Dataset(BCS_GEOM / 'CF0360x6C_CF0360x6C-Pfafstetter.nc4') as d:
    land = np.asarray(d['typ'][:]) == 100
    i_indg = np.asarray(d['i_indg'][:])[land]
    j_indg = np.asarray(d['j_indg'][:])[land]
    tile_lat = np.asarray(d['com_lat'][:])[land]
    tile_lon = np.asarray(d['com_lon'][:])[land]
N_TILE = int(land.sum())
N_CELL = 6 * N_FACE * N_FACE
TILE_CELL = (j_indg - 1) * N_FACE + i_indg - 1           # 0-based cube-cell index of each land tile

with nc.Dataset(CATCH_PARAMS) as d:
    POROS = np.asarray(d['POROS'][:], dtype=np.float64)
assert POROS.size == N_TILE

STRIDE, RES = 20, 1.0 / 120.0
mm = np.memmap(BCS_GEOM / 'rst' / 'CF0360x6C.rst', dtype='<i4', mode='r').reshape(21600, 43202)
CELL_RASTER = np.array(mm[::STRIDE, 1:-1:STRIDE])         # rows S -> N; values = 1-based cell number
CR_LAT_E = -90 + RES * STRIDE * np.arange(CELL_RASTER.shape[0] + 1)
CR_LON_E = -180 + RES * STRIDE * np.arange(CELL_RASTER.shape[1] + 1)
HAS_LAND = np.bincount(TILE_CELL, minlength=N_CELL) > 0
print(f'{N_TILE:,} land tiles in {HAS_LAND.sum():,} cube cells; porosity {POROS.min():.2f}-{POROS.max():.2f}')


def worst_tile_per_cell(tile_values):
    # For each cube cell: the LARGEST value among its land tiles (NaN for cells without land).
    out = np.zeros(N_CELL)
    np.maximum.at(out, TILE_CELL, tile_values)
    out[~HAS_LAND] = np.nan
    return out

## 3. Scan all cycles: flag counts per tile (cached)

Per cycle, we read `inst3_1d_lndfcstana_Nt` (FCST, ANA and FCST ensemble std of SFMC, RZMC, TSURF),
`catch_progn_incr` (dW and the TC increments) and `ObsFcstAna` (normalized innovations). Then:

- we add 1 to each tile's count for every criterion it meets in that cycle;
- for `C_pin`, we keep a running count of consecutive pinned cycles per tile and remember the longest run;
- for the event table, we keep every flagged (tile, cycle) of `A_RZ`, `B_dW`, `A_TS` and `B_TC` with its size.

Fill values are NaN and never flag.

In [ ]:
CRITERIA = ['A_RZ', 'A_SF', 'A_TS', 'A_TS_snow', 'B_dW', 'B_RZ', 'B_SF', 'B_TC', 'D_obs']
count_cache = CACHE_DIR / 'flag_counts_v3.npz'
event_cache = CACHE_DIR / 'flag_events_v3.csv'


def stamp_of(f):
    return os.path.basename(f).split('.')[-2]              # e.g. 20190515_0300z


def cycle_files(pattern):
    files = sorted(glob.glob(str(OUT / 'cat' / 'ens_avg' / 'Y*' / 'M*' / pattern)))
    return {stamp_of(f): f for f in files if START <= stamp_of(f)[:8] <= END}


if count_cache.exists() and event_cache.exists() and not FORCE_RECOMPUTE:
    C = dict(np.load(count_cache))
    EVENTS = pd.read_csv(event_cache)
    print('using cached counts')
else:
    t0 = time.time()
    fa_files = cycle_files(f'{EXP}.inst3_1d_lndfcstana_Nt.*.nc4')
    in_files = cycle_files(f'{EXP}.catch_progn_incr.*.nc4')
    ofa_files = {stamp_of(f): f for f in sorted(glob.glob(str(OUT / 'ana' / 'ens_avg' / 'Y*' / 'M*'
                                                            / f'{EXP}.ens_avg.ldas_ObsFcstAna.*.nc4')))}
    stamps = sorted(set(fa_files) & set(in_files))
    C = {k: np.zeros(N_TILE, np.int32) for k in CRITERIA}
    C['wet_flag'] = np.zeros(N_TILE, bool)                  # any wetting A_RZ / B_dW / B_RZ this month
    run = np.zeros(N_TILE, np.int32)
    C['pin_maxrun'] = np.zeros(N_TILE, np.int32)
    events = []

    def rd(d, v):
        return np.ma.filled(d[v][0].astype(np.float64), np.nan)

    for s in stamps:
        with nc.Dataset(fa_files[s]) as d:
            dSF = rd(d, 'SFMC_ANA') - rd(d, 'SFMC_FCST'); sSF = rd(d, 'SFMC_FCST_ENSSTD')
            dRZ = rd(d, 'RZMC_ANA') - rd(d, 'RZMC_FCST'); sRZ = rd(d, 'RZMC_FCST_ENSSTD')
            dTS = rd(d, 'TSURF_ANA') - rd(d, 'TSURF_FCST'); sTS = rd(d, 'TSURF_FCST_ENSSTD')
            rz_ana = rd(d, 'RZMC_ANA')
        with nc.Dataset(in_files[s]) as d:
            dW = rd(d, 'SRFEXC_INCR') + rd(d, 'RZEXC_INCR') - rd(d, 'CATDEF_INCR')
            dTC = np.nanmax(np.abs([rd(d, 'TCFSAT_INCR'), rd(d, 'TCFTRN_INCR'), rd(d, 'TCFWLT_INCR')]), axis=0)
            snow = np.abs(np.nan_to_num(rd(d, 'WESNN1_INCR') + rd(d, 'WESNN2_INCR') + rd(d, 'WESNN3_INCR'))) > 0

        with np.errstate(invalid='ignore', divide='ignore'):
            zRZ, zSF, zTS = np.abs(dRZ) / sRZ, np.abs(dSF) / sSF, np.abs(dTS) / sTS
            flags = {
                'A_RZ': (sRZ > 0) & (zRZ > Z_MAX) & (np.abs(dRZ) > MIN_DRZ_A),
                'A_SF': (sSF > 0) & (zSF > Z_MAX) & (np.abs(dSF) > MIN_DSF_A),
                'A_TS': (sTS > 0) & (zTS > Z_MAX) & (np.abs(dTS) > MIN_DTS_A) & ~snow,
                'A_TS_snow': (zTS > Z_MAX) & (np.abs(dTS) > MIN_DTS_A) & snow,
                'B_dW': np.abs(dW) > MAX_DW,
                'B_RZ': np.abs(dRZ) > MAX_DRZ,
                'B_SF': np.abs(dSF) > MAX_DSF,
                'B_TC': dTC > MAX_DTC,
            }
            pinned = ((POROS - rz_ana) < PIN_TOL) & (sRZ < PIN_MAX_SPREAD)
        for k, f in flags.items():
            C[k] += f
        C['wet_flag'] |= (flags['A_RZ'] & (dRZ > 0)) | (flags['B_dW'] & (dW > 0)) | (flags['B_RZ'] & (dRZ > 0))
        run = np.where(pinned, run + 1, 0)
        C['pin_maxrun'] = np.maximum(C['pin_maxrun'], run)

        # events for the table (value = the flagged quantity, signed where meaningful)
        for k, val, z in [('A_RZ', dRZ, zRZ), ('B_dW', dW, None), ('A_TS', dTS, zTS), ('B_TC', dTC, None)]:
            idx = np.flatnonzero(flags[k])
            events.append(pd.DataFrame({'criterion': k, 'cycle': s, 'tile': idx + 1, 'value': val[idx],
                                        'z': z[idx] if z is not None else np.nan}))

        # group D: extreme normalized innovations of assimilated obs, counted on their tile
        if s in ofa_files:
            with nc.Dataset(ofa_files[s]) as d:
                if d.dimensions['n_obs'].size:
                    g = lambda v: np.ma.filled(d[v][:].astype(np.float64), np.nan)
                    norm = (g('obs') - g('fcst')) / np.sqrt(g('obsvar') + g('fcstvar'))
                    bad = (d['assim_flag'][:] == 1) & (np.abs(norm) > MAX_NORM_INNOV)
                    np.add.at(C['D_obs'], d['tilenum'][:][bad] - 1, 1)

    EVENTS = pd.concat(events, ignore_index=True)
    np.savez_compressed(count_cache, **C)
    EVENTS.to_csv(event_cache, index=False)
    print(f'{len(stamps)} cycles scanned in {time.time() - t0:.0f} s')

C['C_pin'] = (C['pin_maxrun'] >= PIN_CYCLES).astype(np.int32)
C['C_pin_DA'] = (C['C_pin'].astype(bool) & C['wet_flag'].astype(bool)).astype(np.int32)
N_CYC = 247
summary = pd.DataFrame({k: {'tiles flagged at least once': int((C[k] > 0).sum()),
                            'tile-cycles flagged': int(C[k].sum())} for k in CRITERIA + ['C_pin', 'C_pin_DA']}).T
summary

## 4. Maps

Every map shows, per cube cell, the **number of flagged cycles of the worst land tile in that cell** (out of 247),
except group C, where a cell is either pinned or not. Grey = land with no flag, white = ocean. Robinson, north of 60S.

In [ ]:
COUNT_LEVELS = [1, 2, 3, 5, 10, 20, 50, 100, 248]


def count_panels(keys, titles, suptitle, fname, ncol=2, levels=COUNT_LEVELS, cmap='plasma_r',
                 cbar_label='flagged cycles of the worst tile in the cell (of 247)'):
    n = len(keys)
    nrow = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(8 * ncol, 4.9 * nrow + 1.2),
                            subplot_kw={'projection': ccrs.Robinson()}, constrained_layout=True)
    axs = np.atleast_1d(axs).ravel()
    cm = plt.get_cmap(cmap, len(levels) - 1)
    norm = mcolors.BoundaryNorm(levels, cm.N)
    for ax, k, title in zip(axs, keys, titles):
        cellv = worst_tile_per_cell(C[k].astype(float))
        field = cellv[CELL_RASTER - 1]
        field = np.where(field > 0, field, np.nan)                 # 0 -> grey land underneath
        ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
        mesh = ax.pcolormesh(CR_LON_E, CR_LAT_E, field, cmap=cm, norm=norm, transform=ccrs.PlateCarree(),
                             rasterized=True, zorder=1)
        ax.coastlines(linewidth=0.3, zorder=2)
        extra = f', {int(C[k].sum()):,} tile-cycles' if k in CRITERIA else ''
        ax.set_title(f'{title}\n{int((C[k] > 0).sum()):,} tiles flagged{extra}', fontsize=10.5)
    for ax in axs[n:]:
        ax.set_visible(False)
    fig.colorbar(mesh, ax=axs[:n].tolist(), orientation='horizontal', shrink=0.5, pad=0.02, label=cbar_label)
    fig.suptitle(suptitle, fontsize=13)
    fig.savefig(FIG_DIR / fname, dpi=150)
    plt.show()


PERIOD = '1–31 May 2019, 247 cycles'
DEF = 'Δx = analysis − forecast (ensemble mean) at a 3-hourly cycle;  σf(x) = forecast ensemble std'

### 4a. Group A: increments large compared with the forecast spread

In [ ]:
count_panels(['A_RZ', 'A_SF', 'A_TS'],
             [f'A_RZ: |ΔRZMC| > {Z_MAX:g}·σf(RZMC) and |ΔRZMC| > {MIN_DRZ_A} m³/m³',
              f'A_SF: |ΔSFMC| > {Z_MAX:g}·σf(SFMC) and |ΔSFMC| > {MIN_DSF_A} m³/m³',
              f'A_TS: |ΔTSURF| > {Z_MAX:g}·σf(TSURF) and |ΔTSURF| > {MIN_DTS_A:g} K, no snow update'],
             f'allsp, {PERIOD}: cycles where the increment exceeds {Z_MAX:g} forecast ensemble std\n{DEF}',
             'map_A_spread_relative.png')

### 4a'. TSURF flags that come with a snow update

In [ ]:
count_panels(['A_TS_snow'],
             [f'A_TS_snow: |ΔTSURF| > {Z_MAX:g}·σf(TSURF) and |ΔTSURF| > {MIN_DTS_A:g} K, in a cycle with a SWE increment'],
             f'allsp, {PERIOD}: surface-temperature increments that come with a snow (MODIS SCF) update\n' + DEF,
             'map_A_TS_snow.png', ncol=1)

### 4b. Group B: physically large increments

In [ ]:
count_panels(['B_dW', 'B_RZ', 'B_SF', 'B_TC'],
             [f'B_dW: |dW| > {MAX_DW:g} mm   (dW = ΔSRFEXC + ΔRZEXC − ΔCATDEF, profile water)',
              f'B_RZ: |ΔRZMC| > {MAX_DRZ} m³/m³',
              f'B_SF: |ΔSFMC| > {MAX_DSF} m³/m³',
              f'B_TC: max(|ΔTCFSAT|, |ΔTCFTRN|, |ΔTCFWLT|) > {MAX_DTC:g} K'],
             f'allsp, {PERIOD}: cycles with a physically large increment in one 3-h update\n{DEF}',
             'map_B_absolute.png')

### 4c. Group C: pinned at saturation

A cell is coloured if one of its land tiles had RZMC_ANA within 0.02 m³/m³ of porosity **and** σf(RZMC) < 0.005 for
at least 40 consecutive cycles (5 days). The left panel includes naturally saturated tiles. The right panel keeps only
tiles that also received a **wetting** `A_RZ`, `B_dW` or `B_RZ` increment during the month, i.e. tiles that DA
plausibly pushed to saturation. Colour = the longest pinned run of the worst tile, in days.

In [ ]:
C['pin_days_all'] = np.where(C['C_pin'] > 0, C['pin_maxrun'] / 8.0, 0.0)
C['pin_days_DA'] = np.where(C['C_pin_DA'] > 0, C['pin_maxrun'] / 8.0, 0.0)
count_panels(['pin_days_all', 'pin_days_DA'],
             [f'C_pin: porosity − RZMC_ANA < {PIN_TOL} and σf(RZMC) < {PIN_MAX_SPREAD} for ≥ {PIN_CYCLES // 8} days',
              'C_pin_DA: C_pin and the tile had a wetting A_RZ / B_dW / B_RZ flag in May'],
             f'allsp, {PERIOD}: root zone pinned at saturation with collapsed forecast spread\n'
             'colour = longest pinned run of the worst tile in the cell [days]',
             'map_C_pinned.png', levels=[5, 7, 10, 15, 20, 25, 31], cmap='viridis_r',
             cbar_label='longest consecutive pinned period [days]')

### 4d. Group D (obs space): extreme normalized innovations of assimilated obs

In [ ]:
count_panels(['D_obs'],
             [f'D_obs: assimilated obs with |O − F| / sqrt(obsvar + fcstvar) > {MAX_NORM_INNOV:g}, counted on the obs tile'],
             f'allsp, {PERIOD}: extreme normalized innovations (all species)\n'
             'colour = number of such obs on the worst tile in the cell', 'map_D_obs_extreme_innov.png', ncol=1,
             levels=[1, 2, 3, 5, 10, 20, 50, 100, 1000], cbar_label='obs with |normalized innovation| > 5 (worst tile)')

### 4e. Which kinds of flag, where

One map combining the groups. A cell's colour says which of these the worst tiles in the cell had:
**SM** = a strong soil-moisture flag (A_RZ, B_dW or B_RZ), **T** = a large Catchment temperature increment (B_TC),
**PIN** = C_pin_DA. Each cell falls in one of 7 combinations. A_SF, A_TS and B_SF are **left out** here: they fire
in so many places (tens of thousands of tiles, usually once or twice) that they would hide the strong cases. Their
own maps are above.

In [ ]:
sm = (C['A_RZ'] + C['B_dW'] + C['B_RZ']) > 0
tt = C['B_TC'] > 0
pn = C['C_pin_DA'] > 0
code_tile = sm * 1 + tt * 2 + pn * 4
code_cell = worst_tile_per_cell(sm.astype(float)) * 1 + worst_tile_per_cell(tt.astype(float)) * 2 \
            + worst_tile_per_cell(pn.astype(float)) * 4
names = {1: 'SM only', 2: 'T only', 3: 'SM + T', 4: 'PIN only', 5: 'SM + PIN', 6: 'T + PIN', 7: 'SM + T + PIN'}
colors = {1: '#1f77b4', 2: '#ff7f0e', 3: '#9467bd', 4: '#2ca02c', 5: '#17becf', 6: '#bcbd22', 7: '#d62728'}
cm = mcolors.ListedColormap([colors[k] for k in range(1, 8)])
field = code_cell[CELL_RASTER - 1]
field = np.where(field > 0, field, np.nan)

fig = plt.figure(figsize=(15, 7.5), constrained_layout=True)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
ax.pcolormesh(CR_LON_E, CR_LAT_E, field, cmap=cm, vmin=0.5, vmax=7.5, transform=ccrs.PlateCarree(),
              rasterized=True, zorder=1)
ax.coastlines(linewidth=0.3, zorder=2)
counts = pd.Series(code_cell[np.isfinite(code_cell) & (code_cell > 0)]).value_counts()
ax.legend(handles=[mpatches.Patch(color=colors[k], label=f'{names[k]} ({int(counts.get(k, 0)):,} cells)')
                   for k in range(1, 8)], loc='lower left', fontsize=9)
ax.set_title(f'allsp, {PERIOD}: cube cells with a strongly flagged tile, by kind of flag\n'
             f'SM = A_RZ (|ΔRZMC| > 3σf and > {MIN_DRZ_A}), B_dW (|dW| > {MAX_DW:g} mm) or B_RZ (|ΔRZMC| > {MAX_DRZ}); '
             f'T = B_TC (ΔTC > {MAX_DTC:g} K); PIN = C_pin_DA (pinned at saturation after DA wetting)\n'
             '(the frequent weaker flags A_SF, A_TS, B_SF are not included here)', fontsize=10.5)
fig.savefig(FIG_DIR / 'map_E_flag_kinds.png', dpi=150)
plt.show()

## 5. Where are the flags concentrated? Latitude bands and cycle hours

The share of land tiles with at least one flag per latitude band, and the number of flagged tile-cycles per
synoptic hour (the hour says which satellites overpass, e.g. SMAP at 6 am/6 pm local time).

In [ ]:
bands = [(-60, -30), (-30, 0), (0, 30), (30, 45), (45, 55), (55, 90)]
rows = []
for k in CRITERIA + ['C_pin', 'C_pin_DA']:
    r = {'criterion': k}
    for lo, hi in bands:
        m = (tile_lat >= lo) & (tile_lat < hi)
        r[f'{lo}..{hi}'] = np.mean(C[k][m] > 0)
    rows.append(r)
display(pd.DataFrame(rows).set_index('criterion').style.format('{:.2%}').set_caption(
    'share of land tiles flagged at least once, by latitude band'))

EVENTS['hour'] = EVENTS.cycle.str[9:11]
EVENTS.pivot_table(index='criterion', columns='hour', values='tile', aggfunc='count', fill_value=0)

## 6. The largest events, and which obs were assimilated nearby

The top events of each criterion by size (`A_RZ`, `A_TS`: largest z; `B_dW`: largest |dW|; `B_TC`: largest ΔTC), with the
obs **assimilated within ±0.5° of the tile at the same cycle**: count per sensor, mean O − F, and the largest
|normalized innovation|. A large increment next to huge normalized innovations of one sensor (e.g. SMOS Tb over a
floodplain) is a QC/obs lead. A large increment with ordinary innovations points at the filter.

In [ ]:
SPECIES_GROUP = {1: 'SMOS', 2: 'SMOS', 3: 'SMOS', 4: 'SMOS', 5: 'SMAP', 6: 'SMAP', 7: 'SMAP', 8: 'SMAP',
                 9: 'MODIS', 10: 'MODIS', 11: 'CYGNSS', 12: 'HSAF', 13: 'HSAF', 14: 'HSAF'}
# (species index -> sensor for THIS run; from its LDASsa_SPECIAL_inputs_ensupd.nml, see the OFA-maps notebook)

rank = {'A_RZ': 'z', 'A_TS': 'z', 'B_dW': 'absval', 'B_TC': 'absval'}
EVENTS['absval'] = EVENTS.value.abs()
top = pd.concat([EVENTS[EVENTS.criterion == k].nlargest(N_TOP, col) for k, col in rank.items()], ignore_index=True)
top['lat'] = tile_lat[top.tile - 1]
top['lon'] = tile_lon[top.tile - 1]

ofa_cache = {}
def obs_near(cycle, lat, lon, box=0.5):
    if cycle not in ofa_cache:
        f = glob.glob(str(OUT / 'ana' / 'ens_avg' / 'Y*' / 'M*' / f'{EXP}.ens_avg.ldas_ObsFcstAna.{cycle}.nc4'))
        with nc.Dataset(f[0]) as d:
            g = lambda v: np.ma.filled(d[v][:].astype(np.float64), np.nan)
            ofa_cache[cycle] = pd.DataFrame({'sp': d['species'][:], 'assim': d['assim_flag'][:], 'lat': g('lat'),
                                             'lon': g('lon'), 'omf': g('obs') - g('fcst'),
                                             'norm': (g('obs') - g('fcst')) / np.sqrt(g('obsvar') + g('fcstvar'))})
    o = ofa_cache[cycle]
    o = o[(o.assim == 1) & (np.abs(o.lat - lat) <= box) & (np.abs(o.lon - lon) <= box)]
    if o.empty:
        return 'no assimilated obs within ±0.5°'
    parts = []
    for sensor, x in o.groupby(o.sp.map(SPECIES_GROUP)):
        parts.append(f'{sensor}: n={len(x)}, O−F={x.omf.mean():+.3g}, max|norm|={x.norm.abs().max():.1f}')
    return '; '.join(parts)

top['obs within ±0.5° (assimilated)'] = [obs_near(c, la, lo) for c, la, lo in zip(top.cycle, top.lat, top.lon)]
top.to_csv(CACHE_DIR / 'top_events_with_obs.csv', index=False)
pd.set_option('display.max_colwidth', 200)
top[['criterion', 'cycle', 'tile', 'lat', 'lon', 'value', 'z', 'obs within ±0.5° (assimilated)']].round(3)

## 7. Notes

- The thresholds are in the configuration cell. The criteria are screening tools: a flag means "look here", not "wrong".
- σf is the ensemble spread written in the same file as the forecast, i.e. at the time of the update.
- Group D counts obs on the tile they were **assigned** to (see `cf0360_grid_and_tiles.ipynb`, section 6), which is
  not always the tile the obs lies in.